# GCS Emotion Steering

Configuration Variable

In [ ]:
EMOTION = "evil" # change emotion here

Install Dependencies

In [ ]:
!pip install torch transformers numpy matplotlib scikit-learn tqdm --quiet

Imports

In [ ]:
import os
import gc
import csv
import pickle
import numpy as np
from typing import List, Dict, Optional
from dataclasses import dataclass, field

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

Utils & Config

In [ ]:
def clear_gpu_memory():
    """Clear GPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    print("✅ GPU memory cleared")

@dataclass
class Config:
    STEERING_MODEL: str = "meta-llama/Llama-3.1-8B-Instruct"

    # Dynamic concepts
    CONCEPT_POSITIVE: str = EMOTION
    CONCEPT_NEGATIVE: str = "neutral"

    STEER_LAYERS: List[int] = field(default_factory=lambda: list(range(10, 23)))
    SAMPLES_PER_CONCEPT: int = 500
    BATCH_SIZE_HS: int = 4
    MAX_LENGTH: int = 128

    STEERING_STRENGTH: float = 0.1

    # GCS parameters
    M: int = 100
    SUBSAMPLE_RATIO: float = 0.8
    GCS_RANDOM_STATE: int = 42
    SIGMA_LEVELS: List[float] = field(default_factory=lambda: [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 7.0])

    CACHE_DIR: str = "./cache"
    RESULTS_DIR: str = "./results"
    DATASET_DIR: str = "./datasets"

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE: torch.dtype = torch.float16 if torch.cuda.is_available() else torch.float32

cfg = Config()
os.makedirs(cfg.CACHE_DIR, exist_ok=True)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
os.makedirs(cfg.DATASET_DIR, exist_ok=True)

Dataset Loading

In [ ]:
dataset_file = os.path.join(cfg.DATASET_DIR, "dataset.pkl")

if os.path.exists(dataset_file) and os.path.getsize(dataset_file) > 0:
    print(f"📂 Dataset exists at {dataset_file}, loading...")
    with open(dataset_file, 'rb') as f:
        full_dataset = pickle.load(f)

    # Check if the target concept exists in the dataset
    if cfg.CONCEPT_POSITIVE not in full_dataset:
        available_keys = list(full_dataset.keys())
        raise KeyError(f"Concept '{cfg.CONCEPT_POSITIVE}' not found in dataset. Available: {available_keys}")

    concept_data = full_dataset[cfg.CONCEPT_POSITIVE]

    # Extract positive and negative examples
    positive_samples = concept_data.get("positive")
    negative_samples = concept_data.get("negative")

    print(f"✅ Loaded dataset for '{cfg.CONCEPT_POSITIVE}': {len(positive_samples)} positive + {len(negative_samples)} negative samples")

    TRAINING_DATA = {
        "positive": positive_samples,
        "negative": negative_samples
    }
else:
    print(f"🔄 Dataset not found for concept '{cfg.CONCEPT_POSITIVE}'. Please generate dataset.pkl first.")
    TRAINING_DATA = {}

Model Loading

In [ ]:
print(f"📥 Loading steering model: {cfg.STEERING_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(cfg.STEERING_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.STEERING_MODEL,
    torch_dtype=cfg.DTYPE,
    device_map="auto" if cfg.DEVICE == "cuda" else None,
    low_cpu_mem_usage=True,
    offload_folder="./offload" if cfg.DEVICE == "cuda" else None,
    offload_state_dict=True,
)

print(f"✅ Model loaded. Hidden size: {model.config.hidden_size}")

# Store original layers for resetting
original_layers = [layer for layer in model.model.layers]

def reset_model(model, original_layers):
    """Reset model to its original state"""
    for i, layer in enumerate(original_layers):
        model.model.layers[i] = layer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Hidden State Extraction

In [ ]:
def extract_hidden_states(
    texts: List[str],
    model,
    tokenizer,
    layers: List[int],
    batch_size: int = 4,
    max_length: int = 128
) -> Dict[int, torch.Tensor]:
    """Extract activations with step-by-step memory cleanup"""
    all_states = {l: [] for l in layers}

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting hidden states"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                output_hidden_states=True
            )

        hidden_states = outputs.hidden_states

        for layer_idx in layers:
            last_token_states = hidden_states[layer_idx][:, -1, :].float().cpu()
            all_states[layer_idx].append(last_token_states)

        del inputs, outputs, hidden_states
        clear_gpu_memory()

    for layer_idx in layers:
        all_states[layer_idx] = torch.cat(all_states[layer_idx], dim=0)

    return all_states

# Paths to save activations
pos_states_path = os.path.join(cfg.CACHE_DIR, f"pos_states_{cfg.CONCEPT_POSITIVE}.pkl")
neg_states_path = os.path.join(cfg.CACHE_DIR, f"neg_states_{cfg.CONCEPT_POSITIVE}.pkl")

if os.path.exists(pos_states_path) and os.path.exists(neg_states_path):
    print("Loading previously saved activations...")
    with open(pos_states_path, "rb") as f:
        pos_states = pickle.load(f)
    with open(neg_states_path, "rb") as f:
        neg_states = pickle.load(f)
    print("Activations loaded.")
else:
    print("Extracting activations...")
    layers_to_extract = cfg.STEER_LAYERS

    # Dynamically get texts from the dataset
    pos_texts = TRAINING_DATA.get("positive", [])
    neg_texts = TRAINING_DATA.get("negative", [])

    if not pos_texts or not neg_texts:
        raise ValueError(f"❌ No 'positive' or 'negative' lists found for emotion '{cfg.CONCEPT_POSITIVE}' in dataset.pkl")

    print(f"🔍 Extracting activations for POSITIVE '{cfg.CONCEPT_POSITIVE}'...")
    pos_states = extract_hidden_states(pos_texts, model, tokenizer, layers_to_extract, batch_size=cfg.BATCH_SIZE_HS)
    clear_gpu_memory()

    print(f"🔍 Extracting activations for NEGATIVE (control) '{cfg.CONCEPT_NEGATIVE}'...")
    neg_states = extract_hidden_states(neg_texts, model, tokenizer, layers_to_extract, batch_size=cfg.BATCH_SIZE_HS)
    clear_gpu_memory()

    with open(pos_states_path, "wb") as f:
        pickle.dump(pos_states, f)
    with open(neg_states_path, "wb") as f:
        pickle.dump(neg_states, f)
    print("Activations saved.")

print(f"✅ Extracted: pos={pos_states[cfg.STEER_LAYERS[0]].shape}, neg={neg_states[cfg.STEER_LAYERS[0]].shape}")

GCS Vector Computation

In [ ]:
# ========================
# GCS: Gaussian Concept Subspace
# ========================

def compute_gcs_vectors(
    pos_states: Dict[int, torch.Tensor],
    neg_states: Dict[int, torch.Tensor],
    layers: List[int],
    M: int = 100,
    subsample_ratio: float = 0.8,
    random_state: int = 42
) -> Dict[int, Dict[str, np.ndarray]]:
    """True GCS: M sampled classifiers → Gaussian (mu, sigma)"""
    np.random.seed(random_state)
    gcs = {}

    for layer in tqdm(layers, desc="Computing GCS per layer"):
        pos = pos_states[layer].cpu().numpy()
        neg = neg_states[layer].cpu().numpy()

        n_pos, n_neg = len(pos), len(neg)
        subsample_pos = int(n_pos * subsample_ratio)
        subsample_neg = int(n_neg * subsample_ratio)

        weights = []
        for m in range(M):
            # Random subsample (bootstrap-like)
            pos_idx = np.random.choice(n_pos, subsample_pos, replace=True)
            neg_idx = np.random.choice(n_neg, subsample_neg, replace=True)

            X = np.vstack([pos[pos_idx], neg[neg_idx]])
            y = np.array([1] * subsample_pos + [0] * subsample_neg)

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            clf = LogisticRegression(
                max_iter=1000,
                fit_intercept=False,
                C=1.0,
                random_state=random_state + m
            )
            clf.fit(X_scaled, y)
            weights.append(clf.coef_[0])

        weights = np.array(weights)                    # Shape: (M, hidden_dim)
        mu = weights.mean(axis=0)
        sigma = weights.std(axis=0)

        # Normalize the mean vector (as in original GCS)
        mu = mu / (np.linalg.norm(mu) + 1e-8)

        gcs[layer] = {
            'mu': mu,
            'sigma': sigma,
            'samples': weights   # Keep samples just in case
        }
        print(f"  Layer {layer}: μ-norm={np.linalg.norm(mu):.4f}, σ-mean={sigma.mean():.4f}")

    return gcs

# ========================
# Save / Load GCS
# ========================
gcs_file = os.path.join(cfg.CACHE_DIR, f"gcs_vectors_{cfg.CONCEPT_POSITIVE}.pkl")

if os.path.exists(gcs_file) and os.path.getsize(gcs_file) > 0:
    print(f"📂 GCS vectors already exist, loading...")
    with open(gcs_file, 'rb') as f:
        gcs_vectors = pickle.load(f)
else:
    print(f"🧮 Computing GCS vectors for '{cfg.CONCEPT_POSITIVE}' (M={cfg.M})...")
    gcs_vectors = compute_gcs_vectors(
        pos_states, neg_states,
        cfg.STEER_LAYERS,
        M=cfg.M,
        subsample_ratio=cfg.SUBSAMPLE_RATIO,
        random_state=cfg.GCS_RANDOM_STATE
    )
    with open(gcs_file, 'wb') as f:
        pickle.dump(gcs_vectors, f)
    print("✅ GCS vectors saved.")

del pos_states, neg_states
clear_gpu_memory()
print(f"✅ GCS ready for {len(gcs_vectors)} layers")

Steering Vector Sampling

In [ ]:
def sample_steering_vector(gcs_dict: dict, sigma_level: float = 1.0, n_tries: int = 5) -> np.ndarray:
    mu = gcs_dict['mu']
    sigma = gcs_dict['sigma']
    dim = len(mu)

    for _ in range(n_tries):
        # Uniform sampling in ±sigma_level * sigma
        noise = np.random.uniform(-sigma_level, sigma_level, dim) * sigma
        v = mu + noise
        v = v / (np.linalg.norm(v) + 1e-8)
        return v

    # Fallback
    return mu / (np.linalg.norm(mu) + 1e-8)

Steering Layer Implementation

In [ ]:
class SteeringLayer(nn.Module):
    """Steering layer adapted from the original repository"""
    def __init__(
        self,
        target_layer: nn.Module,
        layer_idx: int,
        steering_vector: np.ndarray,
        strength: float = 0.1
    ):
        super().__init__()
        self.target_layer = target_layer
        self.layer_idx = layer_idx
        self.strength = strength
        self.steering_vector_np = steering_vector
        self._steering_vector = None

    def _get_steering_vector(self, device, dtype):
        """Lazy initialization of the steering vector on the correct device"""
        if self._steering_vector is None or self._steering_vector.device != device:
            self._steering_vector = nn.Parameter(
                torch.tensor(self.steering_vector_np, dtype=dtype, device=device),
                requires_grad=False
            )
        return self._steering_vector

    def forward(self, *args, **kwargs):
        original_output = self.target_layer(*args, **kwargs)

        if isinstance(original_output, tuple):
            hidden_states = original_output[0]
        else:
            hidden_states = original_output

        last_token = hidden_states[:, -1:, :]
        steering_vector = self._get_steering_vector(last_token.device, last_token.dtype)

        scale = last_token.abs().mean() / (steering_vector.abs().mean() + 1e-8)
        steering_term = (steering_vector * scale).to(last_token.dtype)

        modified_last = last_token * (1 - self.strength) + self.strength * steering_term
        hidden_states[:, -1:, :] = modified_last

        if isinstance(original_output, tuple):
            return (hidden_states,) + original_output[1:]
        return hidden_states


def apply_steering_to_model(
    model: nn.Module,
    steering_vectors: Dict[int, np.ndarray],
    strength: float,
    layers: List[int]
) -> nn.Module:
    """Wraps specified model layers in SteeringLayer"""
    for layer_idx in layers:
        if layer_idx in steering_vectors:
            original_layer = model.model.layers[layer_idx]
            model.model.layers[layer_idx] = SteeringLayer(
                original_layer,
                layer_idx,
                steering_vectors[layer_idx],
                strength
            )
    return model

print("✅ SteeringLayer defined")

Pre-Generation Memory Clear

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
print("✅ GPU memory cleared before generation test")

Testing & Generation

In [ ]:
TEST_PROMPTS = [
    "Tell a short story about a person's choice.",
    "Write a movie review in 2-3 sentences.",
    "What do you think about the importance of moral principles?",
    "What would you do if you had one free day with no obligations?",
    "If you could instantly learn one skill without effort, what would it be and why?",
    "Invent a new holiday and describe how people celebrate it."
]

RESULT_CSV = "./generation_results.csv"

def get_existing_prompts(csv_path: str) -> set:
    existing = set()
    if os.path.exists(csv_path):
        with open(csv_path, 'r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                existing.add(row['prompt'])
    return existing

def save_results_to_csv(results_list: List[Dict], csv_path: str, mode: str = 'a') -> None:
    file_exists = os.path.exists(csv_path)
    with open(csv_path, mode, newline='', encoding='utf-8') as f:
        fieldnames = ['prompt_id', 'prompt', 'strength', 'sigma_level', 'response', 'response_length']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists or mode == 'w':
            writer.writeheader()
        writer.writerows(results_list)

def generate_with_steering(
    prompt: str,
    model,
    tokenizer,
    gcs_vectors: Optional[Dict] = None,
    strength: float = 0.0,
    sigma_level: float = 1.0,
    layers: List[int] = None,
    max_new_tokens: int = 100,
    temperature: float = 0.7
) -> str:
    """Generate text with GCS support"""
    reset_model(model, original_layers)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if gcs_vectors and strength != 0 and layers:
        # GCS: Sample a vector for this run
        sampled_vectors = {}
        for layer_idx in layers:
            if layer_idx in gcs_vectors:
                v = sample_steering_vector(gcs_vectors[layer_idx], sigma_level=sigma_level)
                sampled_vectors[layer_idx] = v

        apply_steering_to_model(model, sampled_vectors, strength, layers)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if generated.startswith(prompt):
        generated = generated[len(prompt):].strip()

    return generated

# ========================
# Main GCS Testing Block
# ========================
existing_prompts = get_existing_prompts(RESULT_CSV)
total_tested = 0

for prompt_idx, prompt in enumerate(TEST_PROMPTS, start=1):
    if prompt in existing_prompts:
        print(f"⏭️ Prompt {prompt_idx} already exists, skipping.")
        continue

    print(f"\n📝 Prompt {prompt_idx}: '{prompt[:60]}...'")
    print("-" * 90)

    prompt_results = []
    for sigma_level in tqdm(cfg.SIGMA_LEVELS, desc=f"Prompt {prompt_idx}"):
        response = generate_with_steering(
            prompt, model, tokenizer,
            gcs_vectors=gcs_vectors,
            strength=cfg.STEERING_STRENGTH,
            sigma_level=sigma_level,
            layers=cfg.STEER_LAYERS,
            max_new_tokens=80
        )

        prompt_results.append({
            'prompt_id': prompt_idx,
            'prompt': prompt,
            'strength': cfg.STEERING_STRENGTH,
            'sigma_level': sigma_level,
            'response': response,
            'response_length': len(response.split())
        })

        print(f"  🔹 σ = {sigma_level:+.1f} | {response[:120]}...")

    save_results_to_csv(prompt_results, RESULT_CSV, mode='a')
    existing_prompts.add(prompt)
    total_tested += 1
    print()

print(f"✅ GCS testing finished! Tested {total_tested} new prompts.")
print(f"Results → {RESULT_CSV}")